===============================================================================================

$$\text{This script Reproduce the matched feedback and constant-flow comparison }$$
$$\text{Avulundiah Edwin Phiri}$$
$$\text{Department of Mathematics, Copperbelt University, Kitwe, Zambia}$$
$$\text{31/07/2026}$$

==============================================================================================


In [2]:
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq, minimize_scalar

OUTPUT = Path(__file__).resolve().parent
DELTA, GAMMA, KAPPA, LAMBDA, EXPONENT = 0.15, 1.0, 0.30, 0.35, 5.0
S0, X0, EPSILON, T_END = 0.90, 0.20, 0.40, 30.0


def rhs(t, y, feedback):
    substrate, mineral = y
    growth = substrate / (KAPPA + substrate)
    exchange = (1.0 + X0 - mineral) ** EXPONENT if feedback else 1.0
    return [
        DELTA * exchange * (1.0 - substrate) - GAMMA * growth * mineral,
        (growth - LAMBDA) * mineral,
    ]


def closure_event(t, y, feedback):
    return 1.0 + X0 - y[1]


closure_event.terminal = True
closure_event.direction = -1


def simulation(feedback):
    result = solve_ivp(
        rhs, (0, T_END), (S0, X0), args=(feedback,),
        dense_output=True, events=closure_event,
        method="DOP853", rtol=1e-11, atol=1e-13,
    )
    if not result.success or result.t_events[0].size:
        raise RuntimeError("Simulation failed or reached pore closure")

    def permeability(t):
        return (1 + X0 - result.sol(t)[1]) ** EXPONENT

    scan = np.linspace(0, T_END, 4001)
    values = permeability(scan) - EPSILON
    crossings = [
        brentq(lambda t: permeability(t) - EPSILON, a, b, xtol=1e-12)
        for a, b, fa, fb in zip(scan[:-1], scan[1:], values[:-1], values[1:])
        if fa * fb < 0
    ]
    if len(crossings) != 2:
        raise RuntimeError(f"Expected two threshold crossings, got {crossings}")
    j = int(np.argmin(permeability(scan)))
    a, b = scan[max(0, j - 2)], scan[min(len(scan) - 1, j + 2)]
    minimum = minimize_scalar(permeability, bounds=(a, b), method="bounded")
    return result, permeability, crossings, minimum


def main():
    fb, cf = simulation(True), simulation(False)
    for label, item in (("Feedback", fb), ("Constant flow", cf)):
        _, _, crossings, minimum = item
        print(f"{label}: min={minimum.fun:.6f} at t={minimum.x:.6f}; "
              f"crossings={[round(t, 6) for t in crossings]}; "
              f"duration={crossings[1] - crossings[0]:.6f}")

    t = np.linspace(0, 18, 1200)
    blue, orange = "#145c9e", "#d95f02"
    fig, axes = plt.subplots(3, 1, figsize=(7.2, 7.55), sharex=True)
    for item, color, linestyle, label in (
        (fb, blue, "-", "Feedback"),
        (cf, orange, "--", "Constant flow"),
    ):
        result, permeability, crossings, _ = item
        y = result.sol(t)
        axes[0].plot(t, y[0], linestyle, color=color, lw=2, label=label)
        axes[1].plot(t, y[1], linestyle, color=color, lw=2)
        axes[2].plot(t, permeability(t), linestyle, color=color, lw=2)
        axes[2].scatter(crossings, [EPSILON, EPSILON], color=color, s=27, zorder=5)
        axes[2].axvspan(crossings[0], crossings[1], color=color, alpha=0.075)

    axes[2].axhline(EPSILON, color="black", lw=1, ls=":")
    axes[2].annotate(r"$\varepsilon=0.4$", (16.1, EPSILON),
                     xytext=(0, 6), textcoords="offset points", fontsize=10)
    axes[0].legend(loc="upper right", frameon=False, ncol=2)
    axes[0].set_ylabel(r"Substrate $s(\tau)$")
    axes[1].set_ylabel(r"Mineral $x(\tau)$")
    axes[2].set_ylabel(r"Permeability $k(\tau)/k_0$")
    axes[2].set_xlabel(r"Dimensionless time $\tau$")
    axes[0].set_ylim(0, 1.02)
    axes[1].set_ylim(0, .64)
    axes[2].set_ylim(0, 1.03)
    axes[2].set_xlim(0, 18)
    for ax in axes:
        ax.grid(alpha=.17, lw=.65)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(labelsize=10)
    fig.align_ylabels(axes)
    fig.subplots_adjust(hspace=.13, left=.13, right=.98, bottom=.085, top=.97)
    fig.savefig(OUTPUT / "feedback_comparison_revised.pdf", bbox_inches="tight")
    fig.savefig(OUTPUT / "feedback_comparison_revised.png", dpi=450, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    main()


NameError: name '__file__' is not defined